In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# LR loaders
from edge_evaluations.lr_edge_evaluation import load_lr_model, evaluate_model as evaluate_lr_edge

# RF loaders
from edge_evaluations.rf_edge_evalutaion import evaluate_rf_edge

# XGB loaders
from edge_evaluations.xgb_edge_evaluation import evaluate_xgb_edge


# ============================================================
# Directories
# ============================================================

MODEL_DIR = "../outputs/models/"
PRED_DIR  = "../outputs/models/predictions/"
COMPARE_DIR = "../outputs/comparison/"

os.makedirs(COMPARE_DIR, exist_ok=True)

TARGETS = ["Throughput", "Latency", "Loss"]


# ============================================================
# Helper: Load predictions
# ============================================================

def load_predictions(model_prefix, metric, split):
    """
    Loads predicted-vs-actual CSV for LR, RF, or XGB.
    """
    fname = f"{model_prefix}_{metric}_{split}_predictions.csv"
    path = os.path.join(PRED_DIR, fname)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    return df["y_true"], df["y_pred"]


# ============================================================
# Helper: Compute regression metrics
# ============================================================

def compute_regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return rmse, mae, r2


# ============================================================
# Unified Model Comparison
# ============================================================

def generate_model_comparison(feature_cols, power_watts=2.0):

    rows = []

    for metric in TARGETS:

        print(f"\n=================== MODEL COMPARISON: {metric} ===================")

        # ================================================
        # 1. LOAD PREDICTIONS
        # ================================================
        y_true_lr,  y_pred_lr  = load_predictions("lr",  metric, "test")
        y_true_rf,  y_pred_rf  = load_predictions("rf",  metric, "test")
        y_true_xgb, y_pred_xgb = load_predictions("xgb", metric, "test")

        # ================================================
        # 2. PERFORMANCE METRICS
        # ================================================
        lr_rmse,  lr_mae,  lr_r2  = compute_regression_metrics(y_true_lr,  y_pred_lr)
        rf_rmse,  rf_mae,  rf_r2  = compute_regression_metrics(y_true_rf,  y_pred_rf)
        xgb_rmse, xgb_mae, xgb_r2 = compute_regression_metrics(y_true_xgb, y_pred_xgb)

        # ================================================
        # 3. EDGE EFFICIENCY METRICS
        # ================================================

        # Dummy input for LR & RF
        X_sample_np = np.random.randn(1, len(feature_cols))
        X_sample_df = pd.DataFrame([X_sample_np.flatten()], columns=feature_cols)

        # ----- LR -----
        lr_model_path    = os.path.join(MODEL_DIR, f"lr_model_{metric.lower()}.pkl")
        lr_scaler_path   = os.path.join(MODEL_DIR, "lr_scaler.pkl")

        _, scaler, lr_pt_model = load_lr_model(lr_model_path, lr_scaler_path, len(feature_cols))
        lr_edge = evaluate_lr_edge(lr_pt_model, len(feature_cols))

        # ----- RF -----
        rf_model_path = os.path.join(MODEL_DIR, f"rf_model_{metric.lower()}.pkl")
        rf_edge = evaluate_rf_edge(rf_model_path, X_sample_np)

        # ----- XGB -----
        xgb_model_path = os.path.join(MODEL_DIR, f"xgb_model_{metric.lower()}.json")
        xgb_edge = evaluate_xgb_edge(xgb_model_path, X_sample_df, assumed_power_W=power_watts)

        # ================================================
        # 4. Store results
        # ================================================
        rows.append({
            "Metric": metric,
            "Model": "LR",
            "RMSE": lr_rmse,
            "MAE": lr_mae,
            "R2":  lr_r2,
            "Size_MB": lr_edge["Model Size (MB)"],
            "Latency_ms": lr_edge["Latency (ms)"],
            "Energy_J": lr_edge["Energy Estimated (J)"]
        })

        rows.append({
            "Metric": metric,
            "Model": "RF",
            "RMSE": rf_rmse,
            "MAE": rf_mae,
            "R2":  rf_r2,
            "Size_MB": rf_edge["Model Size (MB)"],
            "Latency_ms": rf_edge["Latency (ms)"],
            "Energy_J": rf_edge["Energy Estimated (J)"]
        })

        rows.append({
            "Metric": metric,
            "Model": "XGB",
            "RMSE": xgb_rmse,
            "MAE": xgb_mae,
            "R2":  xgb_r2,
            "Size_MB": xgb_edge["Model Size (MB)"],
            "Latency_ms": xgb_edge["Latency (ms)"],
            "Energy_J": xgb_edge["Energy Estimated (J)"]
        })

    df_result = pd.DataFrame(rows)

    # Save CSV
    csv_path = os.path.join(COMPARE_DIR, "model_comparison.csv")
    df_result.to_csv(csv_path, index=False)
    print(f"\nSaved CSV → {csv_path}")

    # Save LaTeX
    tex_path = os.path.join(COMPARE_DIR, "model_comparison.tex")
    df_result.to_latex(tex_path, index=False, float_format="%.4f")
    print(f"Saved LaTeX → {tex_path}")

    return df_result


In [2]:
from tqdm import tqdm

def generate_model_comparison(feature_cols, power_watts=2.0):

    rows = []

    print("\n==================== Generating Model Comparison Table ====================\n")

    # tqdm loop for metrics
    for metric in tqdm(TARGETS, desc="Metrics", colour="cyan"):

        print(f"\n--- Evaluating Metric: {metric} ---")

        # -------------------------------------------------
        # LOAD PREDICTIONS (with tqdm)
        # -------------------------------------------------
        with tqdm(total=3, desc=f"Loading predictions ({metric})", colour="yellow") as pbar:
            y_true_lr,  y_pred_lr  = load_predictions("lr",  metric, "test");  pbar.update(1)
            y_true_rf,  y_pred_rf  = load_predictions("rf",  metric, "test");  pbar.update(1)
            y_true_xgb, y_pred_xgb = load_predictions("xgb", metric, "test");  pbar.update(1)

        # -------------------------------------------------
        # PERFORMANCE METRICS
        # -------------------------------------------------
        lr_rmse,  lr_mae,  lr_r2  = compute_regression_metrics(y_true_lr,  y_pred_lr)
        rf_rmse,  rf_mae,  rf_r2  = compute_regression_metrics(y_true_rf,  y_pred_rf)
        xgb_rmse, xgb_mae, xgb_r2 = compute_regression_metrics(y_true_xgb, y_pred_xgb)

        # -------------------------------------------------
        # EDGE EVALUATION WITH PROGRESS BAR
        # -------------------------------------------------
        print(f"\nEvaluating Edge Efficiency for {metric}...")
        with tqdm(total=3, desc=f"Edge Eval ({metric})", colour="green") as pbar:

            # Dummy samples
            X_sample_np = np.random.randn(1, len(feature_cols))
            X_sample_df = pd.DataFrame([X_sample_np.flatten()], columns=feature_cols)

            # ----- LR -----
            lr_model_path    = os.path.join(MODEL_DIR, f"lr_model_{metric.lower()}.pkl")
            lr_scaler_path   = os.path.join(MODEL_DIR, "lr_scaler.pkl")

            _, scaler, lr_pt_model = load_lr_model(lr_model_path, lr_scaler_path, len(feature_cols))
            lr_edge = evaluate_lr_edge(lr_pt_model, len(feature_cols))
            pbar.update(1)

            # ----- RF -----
            rf_model_path = os.path.join(MODEL_DIR, f"rf_model_{metric.lower()}.pkl")
            rf_edge = evaluate_rf_edge(rf_model_path, X_sample_np)
            pbar.update(1)

            # ----- XGB -----
            xgb_model_path = os.path.join(MODEL_DIR, f"xgb_model_{metric.lower()}.json")
            xgb_edge = evaluate_xgb_edge(xgb_model_path, X_sample_df, assumed_power_W=power_watts)
            pbar.update(1)

        # -------------------------------------------------
        # Append results
        # -------------------------------------------------
        rows.extend([
            {
                "Metric": metric, "Model": "LR",
                "RMSE": lr_rmse, "MAE": lr_mae, "R2": lr_r2,
                "Size_MB": lr_edge["Model Size (MB)"],
                "Latency_ms": lr_edge["Latency (ms)"],
                "Energy_J": lr_edge["Energy Estimated (J)"]
            },
            {
                "Metric": metric, "Model": "RF",
                "RMSE": rf_rmse, "MAE": rf_mae, "R2": rf_r2,
                "Size_MB": rf_edge["Model Size (MB)"],
                "Latency_ms": rf_edge["Latency (ms)"],
                "Energy_J": rf_edge["Energy Estimated (J)"]
            },
            {
                "Metric": metric, "Model": "XGB",
                "RMSE": xgb_rmse, "MAE": xgb_mae, "R2": xgb_r2,
                "Size_MB": xgb_edge["Model Size (MB)"],
                "Latency_ms": xgb_edge["Latency (ms)"],
                "Energy_J": xgb_edge["Energy Estimated (J)"]
            }
        ])

    # -----------------------------------------------------
    # Final DataFrame
    # -----------------------------------------------------
    df_result = pd.DataFrame(rows)

    csv_path = os.path.join(COMPARE_DIR, "model_comparison.csv")
    df_result.to_csv(csv_path, index=False)
    print(f"\nSaved CSV → {csv_path}")

    tex_path = os.path.join(COMPARE_DIR, "model_comparison.tex")
    df_result.to_latex(tex_path, index=False, float_format="%.4f")
    print(f"Saved LaTeX → {tex_path}")

    return df_result


In [3]:
import pandas as pd
# from model_comparison import generate_model_comparison

df_train = pd.read_csv("../outputs/datasets/train.csv")

EXCLUDE = ["Perf_Avg_Throughput","Perf_Avg_Latency","Perf_Avg_Loss",
           "StartDateTime","EndDateTime","Perf_Filename","ND_Filename","RunDateTime"]

feature_cols = [c for c in df_train.columns if c not in EXCLUDE]

df_comparison = generate_model_comparison(feature_cols)
print(df_comparison)


FileNotFoundError: [Errno 2] No such file or directory: '../outputs/datasets/train.csv'

In [ ]:
from edge_evaluations.comparison_plots import generate_all_comparison_plots
generate_all_comparison_plots(showImage=False)
